# 보고서용: S2 Parser 결과 통계

In [15]:
# scripts/report_s2_parser_00_stats.py (예시)
import json
from collections import Counter

In [16]:
path = "../runs/s2_parser.ir.jsonl"
n_proto = 0
step_counts = []
qcg_counts = []
data_counts = []
warn_counter = Counter()

In [17]:
with open(path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        n_proto += 1
        nodes = r["nodes"]
        step_counts.append(sum(1 for n in nodes if n["type"] == "Step"))
        qcg_counts.append(sum(1 for n in nodes if n["type"] == "QCGate"))
        data_counts.append(sum(1 for n in nodes if n["type"] == "DataAnalysis"))
        for w in r.get("warnings", []):
            warn_counter[w.split()[0]] += 1  # prefix 카운트

In [18]:
print("n_protocols:", n_proto)
print("avg_steps:", sum(step_counts) / len(step_counts))
print("avg_qcgates:", sum(qcg_counts) / len(qcg_counts))
print("avg_data_nodes:", sum(data_counts) / len(data_counts))
print("warnings:", warn_counter)

n_protocols: 46
avg_steps: 15.891304347826088
avg_qcgates: 1.1521739130434783
avg_data_nodes: 0.9565217391304348
warnings: Counter({'added_placeholder_step': 366, 'max_nodes': 7})


In [ ]:
# placeholder가 많으면 → Task Miner는 괜찮아도, Methods 내용 자체가 부족하거나, 프롬프트/MAX_NODES 설정이 빡센 것.

# 보고서용: S2B Evidence Only 결과 통계

In [12]:
# scripts/report_s2b_evidence_00_quality.py
import json
from collections import Counter

path = "../runs/s2b_grounded.evidence_only.ir.jsonl"

n_steps = 0
n_with_ev = 0
domain_counter = Counter()
domain_with_ev = Counter()

In [13]:
with open(path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        domain = r.get("domain") or "unknown"
        nodes = r["nodes"]
        for n in nodes:
            if n.get("type") != "Step":
                continue
            n_steps += 1
            domain_counter[domain] += 1
            if n.get("evidence"):
                n_with_ev += 1
                domain_with_ev[domain] += 1

In [14]:
print("총 Step 수:", n_steps)
print("evidence 있는 Step 수:", n_with_ev, "(비율:", n_with_ev / max(1, n_steps), ")")

print("\n도메인별 coverage:")
for d in sorted(domain_counter.keys()):
    tot = domain_counter[d]
    ok = domain_with_ev[d]
    print(f"  {d:40s}  {ok}/{tot}  ({ok / max(1, tot):.3f})")

총 Step 수: 731
evidence 있는 Step 수: 11 (비율: 0.015047879616963064 )

도메인별 coverage:
  Biochemical & Molecular Functional Analysis  1/64  (0.016)
  Bioimaging Technologies                   1/66  (0.015)
  Cell Biology & Culture                    0/75  (0.000)
  Genomics Technologies                     2/76  (0.026)
  Immunological Techniques                  3/80  (0.037)
  Microbiology & Virology                   3/68  (0.044)
  Model Organism-Specific Techniques        0/60  (0.000)
  Molecular Biology Techniques              0/80  (0.000)
  Neuroscience Methods                      1/82  (0.012)
  Plant Science & Technology                0/80  (0.000)


# 보고서용: IR Evidence Consistency 결과 통계

In [19]:
path = "../eval/ir_evidence_consistency.jsonl"
total = 0
supported = 0

In [20]:
with open(path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        params = r["params"]
        sup = r["check"]["supported_params"]
        uns = r["check"]["unsupported_params"]
        total += len(params)
        supported += len(sup)

In [21]:

print("총 파라미터 수:", total)
print("지원되는 파라미터 수:", supported)
print("지원 비율:", supported / max(1, total))

총 파라미터 수: 1162
지원되는 파라미터 수: 1138
지원 비율: 0.9793459552495697


In [30]:
path = "../eval/ir_evidence_consistency.no_ev.jsonl"
total = 0
supported = 0

In [31]:
with open(path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        params = r["params"]
        sup = r["check"]["supported_params"]
        uns = r["check"]["unsupported_params"]
        total += len(params)
        supported += len(sup)

In [32]:

print("총 파라미터 수:", total)
print("지원되는 파라미터 수:", supported)
print("지원 비율:", supported / max(1, total))

총 파라미터 수: 1162
지원되는 파라미터 수: 1141
지원 비율: 0.9819277108433735


# 보고서용: parameter 근거 품질 결과 통계

In [26]:
import json
import pandas as pd

rows = []
with open("../runs/verify_cov_params.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

# 전체 비율
print(df["verdict"].value_counts(normalize=True))

verdict
supported      0.783333
unsupported    0.128333
ambiguous      0.088333
Name: proportion, dtype: float64


In [28]:
# 프로토콜별 supported 비율
per_proto = df.groupby("protocol_id")["verdict"].value_counts(normalize=True).unstack().fillna(0)
print(per_proto)

verdict            ambiguous  supported  unsupported
protocol_id                                         
Bio-protocol-1010        0.0   1.000000     0.000000
Bio-protocol-1111        0.0   1.000000     0.000000
Bio-protocol-1174        0.0   1.000000     0.000000
Bio-protocol-1213        0.0   0.947368     0.052632
Bio-protocol-1250        0.0   0.894737     0.105263
Bio-protocol-1437        1.0   0.000000     0.000000
Bio-protocol-1537        1.0   0.000000     0.000000
Bio-protocol-1542        0.0   0.937500     0.062500
Bio-protocol-1611        0.0   0.948276     0.051724
Bio-protocol-1716        0.0   0.944444     0.055556
Bio-protocol-1784        0.0   1.000000     0.000000
Bio-protocol-1836        0.0   0.777778     0.222222
Bio-protocol-1861        0.0   0.884615     0.115385
Bio-protocol-2001        0.0   0.818182     0.181818
Bio-protocol-2096        0.0   0.881579     0.118421
Bio-protocol-2219        0.0   0.841270     0.158730
Bio-protocol-2302        0.0   0.809524     0.

In [40]:
pd.set_option('display.max_rows', None)

In [46]:
df[df["protocol_id"] == "Bio-protocol-2096"]

,protocol_id,node_id,param_index,name,value,unit,verdict,evidence_span,llm_raw
0,Bio-protocol-2096,S1,0,Glucose concentration,1.45,%,supported,"Neural stem cells, Cor3–1 were grown in Comple...","{\n ""verdict"": ""supported"",\n ""evidence_..."
1,Bio-protocol-2096,S1,1,penicillin concentration,100.00,units/ml,supported,"Neural stem cells, Cor3–1 were grown in Comple...","{\n ""verdict"": ""supported"",\n ""evidence_..."
2,Bio-protocol-2096,S1,2,streptomycin concentration,100.00,μg/ml,supported,"Neural stem cells, Cor3–1 were grown in Comple...","{\n ""verdict"": ""supported"",\n ""evidence_..."
3,Bio-protocol-2096,S1,3,BSA concentration,0.16,%,supported,"Neural stem cells, Cor3–1 were grown in Comple...","{\n ""verdict"": ""supported"",\n ""evidence_..."
4,Bio-protocol-2096,S1,4,β-Mercapto-ethanol concentration,0.10,mM,supported,"Neural stem cells, Cor3–1 were grown in Comple...","{\n ""verdict"": ""supported"",\n ""evidence_..."
5,Bio-protocol-2096,S1,5,B-27 supplement concentration,1.00,%,supported,"Neural stem cells, Cor3–1 were grown in Comple...","{\n ""verdict"": ""supported"",\n ""evidence_..."
6,Bio-protocol-2096,S1,6,N2 supplement concentration,0.50,%,supported,"Neural stem cells, Cor3–1 were grown in Comple...","{\n ""verdict"": ""supported"",\n ""evidence_..."
7,Bio-protocol-2096,S1,7,mouse EGF concentration,10.00,ng/ml,supported,Before changing the medium or passaging the ce...,"{\n ""verdict"": ""supported"",\n ""evidence_..."
8,Bio-protocol-2096,S1,8,human FGF concentration initial,10.00,ng/ml,supported,Before changing the medium or passaging the ce...,"{\n ""verdict"": ""supported"",\n ""evidence_..."
9,Bio-protocol-2096,S1,9,Laminin concentration,1.00,μg/ml,supported,"Neural stem cells, Cor3–1 were grown in Comple...","{\n ""verdict"": ""supported"",\n ""evidence_..."


In [52]:
# 예: unsupported 이면서 evidence_span 이 빈 문자열인 케이스만 모아보기
unsupported = df[(df["unit"] == "×")]
print(unsupported)

           protocol_id node_id  param_index  \
19   Bio-protocol-2096      S3            4   
318  Bio-protocol-1437      S8            2   
327  Bio-protocol-1437      S8           11   
620  Bio-protocol-1611      S6            0   
621  Bio-protocol-1611      S6            1   

                                       name  value unit      verdict  \
19                   GlutaMAX concentration    1.0    ×  unsupported   
318            objective lens magnification   25.0    ×    ambiguous   
327                           magnification    2.0    ×    ambiguous   
620  beads ratio for large fragment removal    0.2    ×    supported   
621  beads ratio for target library binding    0.8    ×    supported   

                                         evidence_span  \
19                                                       
318                                                      
327                                                      
620  The PCR reaction was purified with bead-based .

In [ ]:
(df["verdict"] == "unsupported")

In [54]:
import json
import pandas as pd

rows = []
with open("../runs/enhanced_cov_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)

# 전체 비율
print(df["verdict"].value_counts(normalize=True))

verdict
supported      0.533333
ambiguous      0.367500
unsupported    0.099167
Name: proportion, dtype: float64


In [55]:
# 프로토콜별 supported 비율
per_proto = df.groupby("protocol_id")["verdict"].value_counts(normalize=True).unstack().fillna(0)
print(per_proto)

verdict            ambiguous  supported  unsupported
protocol_id                                         
Bio-protocol-1010   1.000000   0.000000     0.000000
Bio-protocol-1111   0.000000   1.000000     0.000000
Bio-protocol-1174   1.000000   0.000000     0.000000
Bio-protocol-1213   1.000000   0.000000     0.000000
Bio-protocol-1250   0.000000   0.921053     0.078947
Bio-protocol-1437   0.657143   0.228571     0.114286
Bio-protocol-1537   1.000000   0.000000     0.000000
Bio-protocol-1542   0.000000   0.875000     0.125000
Bio-protocol-1611   0.000000   0.948276     0.051724
Bio-protocol-1716   1.000000   0.000000     0.000000
Bio-protocol-1784   1.000000   0.000000     0.000000
Bio-protocol-1836   0.000000   0.888889     0.111111
Bio-protocol-1861   1.000000   0.000000     0.000000
Bio-protocol-2001   0.818182   0.181818     0.000000
Bio-protocol-2096   0.000000   0.907895     0.092105
Bio-protocol-2219   1.000000   0.000000     0.000000
Bio-protocol-2302   1.000000   0.000000     0.

In [57]:
# 예: unsupported 이면서 evidence_span 이 빈 문자열인 케이스만 모아보기
unsupported = df[(df["verdict"] == "unsupported")]
unsupported.head(50)

,protocol_id,node_id,param_index,name,value,unit,verdict,evidence_span,llm_raw
19,Bio-protocol-2096,S3,4,GlutaMAX concentration,1.000,×,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
27,Bio-protocol-2096,S5,4,denaturation time,10.000,seconds,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
29,Bio-protocol-2096,S5,6,annealing time,10.000,seconds,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
31,Bio-protocol-2096,S5,8,extension time,30.000,seconds,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
60,Bio-protocol-2096,S7,18,chromatin sonication cycle,NaN,None,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
61,Bio-protocol-2096,S7,19,chromatin sonication setting,NaN,None,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
68,Bio-protocol-2096,S8,2,protein amount range,NaN,μg,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
89,Bio-protocol-972,S8,2,rabbit anti-human citrullinated histone H3 ant...,0.010,v/v,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
90,Bio-protocol-972,S8,3,goat anti-human MPO antibody dilution,0.025,v/v,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."
91,Bio-protocol-972,S8,4,secondary donkey anti-rabbit IgG Alexa Fluor 5...,0.005,v/v,unsupported,,"{\n ""verdict"": ""unsupported"",\n ""evidenc..."


In [58]:
import torch

print(torch.backends.mps.is_available())
if torch.backends.mps.is_available():
    x = torch.ones(1, device='mps')
    print(x)


True
tensor([1.], device='mps:0')
